[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/tomasvicar/OMM-public/blob/master/cviceni/C2/notebooks/matice.ipynb)

# Infuzní směs v matici: tatáž úloha, jen větší

Notebook k bloku 4 druhého cvičení. Navazuje na [`infuze.ipynb`](https://colab.research.google.com/github/tomasvicar/OMM-public/blob/master/cviceni/C2/notebooks/infuze.ipynb)
a ukazuje větší model: šest přípravků a čtyři sledované veličiny. Spustit se dá samostatně, nic z předchozího
notebooku nepotřebuje.

V tomto výukovém modelu mícháme šest přípravků a sledujeme energii,
bílkoviny, draslík a objem:

| přípravek | cena Kč/l | energie kcal/l | bílkoviny g/l | draslík mmol/l |
|---|---:|---:|---:|---:|
| Glc 10 % | 40 | 400 | 0 | 0 |
| Glc 40 % | 120 | 1 600 | 0 | 0 |
| AK 10 % | 350 | 400 | 100 | 0 |
| Lipid 20 % | 500 | 2 000 | 0 | 0 |
| KCl 7,45 % | 30 | 0 | 0 | 1 000 |
| voda | 15 | 0 | 0 | 0 |

Vak má mít **přesně 2 l**, aspoň 1 800 kcal, aspoň 70 g bílkovin, draslík mezi
60 a 100 mmol a nejvýš 0,5 l tukové emulze.

## Co přečíst společně

Sloupec matice je přípravek, řádek sledovaná veličina. Například `A[0] @ x`
je celková energie. Pro objem použijeme rovnost, pro ostatní požadavky
příslušnou nerovnost. **Nic nedoplňujete** — spusťte hotový model a kontrolu.

---

*Čísla, se kterými se notebook porovnává, počítají skripty [`kod/infuze_matice.py`](https://github.com/tomasvicar/OMM-public/blob/master/cviceni/C2/kod/infuze_matice.py). Notebook sám staví [`kod/postav_notebooky.py`](https://github.com/tomasvicar/OMM-public/blob/master/cviceni/C2/kod/postav_notebooky.py) — ruční úprava `.ipynb` se při dalším spuštění přepíše.*

In [ ]:
try:
    import cvxpy as cp
except ImportError:
    %pip install -q cvxpy
    import cvxpy as cp

In [ ]:
import cvxpy as cp
import numpy as np

## Data úlohy

Cena je vektor délky 6, živiny jsou řádky matice 4 × 6. Sloupec = přípravek,
řádek = živina.

In [ ]:
NAZVY = ["Glc 10%", "Glc 40%", "AK 10%", "Lipid 20%", "KCl 7,45%", "voda"]
CENA = np.array([40.0, 120.0, 350.0, 500.0, 30.0, 15.0])   # Kč za litr

# řádky = živiny, sloupce = přípravky
# energie [kcal/l], bílkoviny [g/l], draslík [mmol/l], objem [l/l]
A = np.array([
    [400.0, 1600.0, 400.0, 2000.0,    0.0, 0.0],
    [  0.0,    0.0, 100.0,    0.0,    0.0, 0.0],
    [  0.0,    0.0,   0.0,    0.0, 1000.0, 0.0],
    [  1.0,    1.0,   1.0,    1.0,    1.0, 1.0],
])
ENERGIE, BILKOVINY = 1800.0, 70.0
DRASLIK_MIN, DRASLIK_MAX = 60.0, 100.0
OBJEM_VAKU, LIPID_MAX = 2.0, 0.5

print(f"proměnných: {len(CENA)}, živin: {A.shape[0]}, matice A má tvar {A.shape}")

## Model — tytéž tři řádky

`cp.Variable`, `cp.Minimize(CENA @ x)`, seznam omezení. Jediné, co přibylo, jsou
omezení, která u dvou roztoků nebyla: horní mez draslíku, rovnost na objem
a strop lipidů. Omezení se drží ve slovníku, aby se u nich daly vypsat stínové
ceny i s názvem.

In [ ]:
x = cp.Variable(6, nonneg=True)
omezeni = [A[0] @ x >= ENERGIE,
           A[1] @ x >= BILKOVINY,
           A[2] @ x >= DRASLIK_MIN, A[2] @ x <= DRASLIK_MAX,
           A[3] @ x == OBJEM_VAKU,
           x[3] <= LIPID_MAX]
uloha = cp.Problem(cp.Minimize(CENA @ x), omezeni)
uloha.solve(solver=cp.CLARABEL, tol_gap_abs=1e-11, tol_gap_rel=1e-11, tol_feas=1e-11)
assert uloha.status == "optimal", uloha.status
x_vak = np.asarray(x.value).ravel()
cena_vak = float(uloha.value)
print("složení vaku:")
for nazev, objem in zip(NAZVY, x_vak):
    print(f"  {nazev:10s} {objem:7.4f} l")
print(f"cena = {cena_vak:.2f} Kč")

## Kontrola dosazením zpět — pořád stejná

Šest proměnných místo dvou nic nemění: řešení se dosadí do **úplného** seznamu
požadavků a porovná se s ním. Kontrola je tady dokonce důležitější než u dvou
roztoků — přípustnou oblast si už nikdo nenakreslí a oko ji nezkontroluje.

In [ ]:
print(f"energie   {A[0] @ x_vak:8.1f} kcal (aspoň {ENERGIE})")
print(f"bílkoviny {A[1] @ x_vak:8.1f} g    (aspoň {BILKOVINY})")
print(f"draslík   {A[2] @ x_vak:8.1f} mmol ({DRASLIK_MIN} až {DRASLIK_MAX})")
print(f"objem     {A[3] @ x_vak:8.2f} l    (přesně {OBJEM_VAKU})")

assert A[0] @ x_vak >= ENERGIE - 1e-4, "málo energie"
assert A[1] @ x_vak >= BILKOVINY - 1e-4, "málo bílkovin"
assert DRASLIK_MIN - 1e-4 <= A[2] @ x_vak <= DRASLIK_MAX + 1e-4, "draslík mimo meze"
assert abs(A[3] @ x_vak - OBJEM_VAKU) < 1e-6, "vak nemá dva litry"
assert x_vak[3] <= LIPID_MAX + 1e-6, "překročen strop lipidů"
assert np.all(x_vak >= -1e-6), "záporný objem"
assert abs(cena_vak - 364.67) < 0.01, "cena se rozešla s číslem ze skriptu (364,67 Kč)"
print("\nkontrola dosazením zpět prošla, cena sedí na 364,67 Kč")

## Shrnutí

Více proměnných mění velikost dat, ale postup zůstává:
formulace → řešení → kontrola podle zadání.